# RetainIQ: Customer Churn Prediction and Retention

**IBM project submission | Prepared by: Atasi Mahata**

## Business problem
Subscription businesses lose recurring revenue when customers cancel. Customer success teams need an early-warning system that identifies customers at elevated risk, explains the signals behind each prediction, and suggests a proportionate retention action.

**Objective:** estimate each customer's probability of churn from account, billing, engagement, payment, and support data; rank customers for outreach; and estimate the potential economics of retention offers.

**Decision supported:** which customers should the team contact first, what issue should it address, and what is the expected offer cost relative to revenue potentially retained? The model is a prioritization aid, not an automated cancellation or offer decision.

## Project workflow and setup

This submission is a self-contained notebook: data generation, feature engineering, preprocessing, model training, evaluation, explanations, retention offers, and campaign economics are implemented in the cells below. It does not import code from the repository's `src/` folder.

Use Python 3.11. Install the notebook dependencies in the active environment before running the cells:

```bash
pip install pandas numpy scikit-learn xgboost lightgbm shap matplotlib seaborn joblib
```

A full four-model, five-fold training run can take several minutes. The notebook trains the models and saves the selected model plus preprocessing pipeline under `models/churn_model.pkl` in its working directory.

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, precision_recall_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import shap

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')

# Write generated artifacts beside the notebook, or in the project root if run there.
WORK_DIR = Path.cwd()
if WORK_DIR.name == 'notebooks' and (WORK_DIR.parent / 'data').exists():
    WORK_DIR = WORK_DIR.parent
DATA_DIR = WORK_DIR / 'data'
MODEL_DIR = WORK_DIR / 'models'
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / 'churn_data.csv'
MODEL_PATH = MODEL_DIR / 'churn_model.pkl'
print(f'Working directory: {WORK_DIR.resolve()}')

## 1. Data and target overview

The project uses a reproducible synthetic CRM dataset of 20,000 customers. It includes subscription plan and contract details, charges, payment behavior, support experience, product usage, satisfaction, referrals, and a binary `churn` target. Missing values and outliers are intentionally included to exercise realistic preprocessing. The generator adds a billing-cycle-end uplift to the simulated churn label; billing day itself is not retained as a predictor.

In [ ]:
def generate_churn_data(n=20_000, seed=42):
    """Generate a reproducible synthetic subscription-customer dataset."""
    rng = np.random.default_rng(seed)
    tenure = np.concatenate([
        rng.integers(1, 6, n // 3),
        rng.integers(6, 24, n // 3),
        rng.integers(24, 72, n - 2 * (n // 3)),
    ])
    rng.shuffle(tenure)
    plans = rng.choice(['Basic', 'Standard', 'Premium'], n, p=[.30, .45, .25])
    plan_price = {'Basic': 29.99, 'Standard': 59.99, 'Premium': 99.99}
    monthly = np.clip(np.array([plan_price[p] for p in plans]) + rng.normal(0, 5, n), 15, 150)
    contract = rng.choice(['Monthly', 'Yearly'], n, p=[.55, .45])
    payment = rng.choice(['Card', 'UPI', 'Bank', 'Wallet'], n, p=[.35, .30, .20, .15])
    autopay = rng.binomial(1, .65, n)
    tickets = np.clip(rng.negative_binomial(2, .6, n), 0, 15)
    resolution = np.clip(rng.gamma(3, 8, n), .5, 96)
    satisfaction = rng.choice([1, 2, 3, 4, 5], n, p=[.10, .15, .30, .30, .15])
    logins = np.clip(rng.poisson(12, n), 0, 60)
    sessions = np.round(logins * rng.uniform(10, 45, n), 1)
    feature_use = np.round(rng.beta(2, 3, n) * 100, 1)
    days_since = np.clip(rng.exponential(15, n).astype(int), 0, 180)
    discount = rng.binomial(1, .40, n)
    price_increase = rng.binomial(1, .25, n)
    failures = rng.choice([0, 1, 2, 3, 4, 5], n, p=[.55, .20, .12, .07, .04, .02])
    referrals = rng.choice(np.arange(11), n, p=[.40, .20, .12, .08, .06, .04, .04, .02, .02, .01, .01])
    signup = pd.to_datetime('2026-10-05') - pd.to_timedelta(tenure * 30, unit='D')

    risk = (
        (contract == 'Monthly') * .30
        + (monthly - 15) / 135 * .20
        + (1 - logins / 60) * .15
        + np.clip(tickets / 15, 0, 1) * .15
        + (5 - satisfaction) / 4 * .12
        + np.clip(failures / 5, 0, 1) * .10
        + price_increase * .10
        + np.clip(days_since / 180, 0, 1) * .08
        + (autopay == 0) * .05
        - discount * .08
        + rng.normal(0, .06, n)
    )
    billing_day = rng.integers(1, 29, n)
    risk = np.clip(risk + (billing_day >= 25) * .05, .01, .98)
    churn = rng.binomial(1, risk)
    churn_rate = churn.mean()
    if churn_rate < .15:
        churn[rng.choice(np.flatnonzero(churn == 0), int((.18 - churn_rate) * n), replace=False)] = 1
    elif churn_rate > .27:
        churn[rng.choice(np.flatnonzero(churn == 1), int((churn_rate - .22) * n), replace=False)] = 0

    frame = pd.DataFrame({
        'customer_id': [f'CUST{i:06d}' for i in range(1, n + 1)],
        'signup_date': signup.strftime('%Y-%m-%d'), 'tenure_months': tenure,
        'plan_type': plans, 'monthly_charges': monthly.round(2),
        'total_charges': np.maximum(0, monthly * tenure + rng.normal(0, 20, n)).round(2),
        'contract_type': contract, 'payment_method': payment, 'auto_pay': autopay,
        'num_support_tickets_30d': tickets, 'avg_resolution_time_hours': resolution.round(1),
        'satisfaction_score': satisfaction, 'login_count_30d': logins,
        'session_minutes_30d': sessions, 'feature_usage_score': feature_use,
        'days_since_last_login': days_since, 'discount_applied': discount,
        'price_increase_last_3m': price_increase, 'num_payment_failures_6m': failures,
        'referral_count': referrals, 'churn': churn,
    })
    outliers = rng.choice(n, int(n * .015), replace=False)
    frame.loc[outliers, 'monthly_charges'] *= rng.uniform(1.5, 3, len(outliers))
    frame.loc[outliers, 'num_support_tickets_30d'] = rng.integers(10, 30, len(outliers))
    for column in ['avg_resolution_time_hours', 'satisfaction_score', 'session_minutes_30d',
                   'feature_usage_score', 'days_since_last_login', 'num_payment_failures_6m']:
        frame.loc[rng.random(n) < .04, column] = np.nan
    return frame

if DATA_PATH.exists():
    customers = pd.read_csv(DATA_PATH, parse_dates=['signup_date'])
    print(f'Loaded existing synthetic dataset: {DATA_PATH}')
else:
    customers = generate_churn_data()
    customers.to_csv(DATA_PATH, index=False)
    customers['signup_date'] = pd.to_datetime(customers['signup_date'])
    print(f'Generated dataset: {DATA_PATH}')

print(f'Rows: {len(customers):,} | Columns: {customers.shape[1]}')
print(f'Observed churn rate: {customers.churn.mean():.1%}')
print(f'Missing cells: {customers.isna().sum().sum():,}')
display(customers.head())

In [ ]:
quality = pd.DataFrame({
    'dtype': customers.dtypes.astype(str),
    'missing_count': customers.isna().sum(),
    'missing_pct': (customers.isna().mean() * 100).round(2),
    'unique_values': customers.nunique(dropna=True),
})
display(quality.sort_values('missing_pct', ascending=False).head(12))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=customers, x='churn', ax=axes[0], color='#2878A0')
axes[0].set(title='Target distribution', xlabel='Churn (0 = retained, 1 = churned)', ylabel='Customers')
sns.histplot(data=customers, x='monthly_charges', hue='churn', bins=35,
             element='step', stat='density', common_norm=False, ax=axes[1])
axes[1].set(title='Monthly charges by churn outcome', xlabel='Monthly charges')
plt.tight_layout()
plt.show()

## 2. Feature engineering and preprocessing

The notebook derives tenure buckets, charges per month, usage and engagement scores, support load, payment reliability, login recency, and loyalty. Its `ColumnTransformer` median-imputes and scales numeric features, most-frequent-imputes and one-hot encodes categories, and imputes/scales binary indicators. The train/holdout split is stratified; each cross-validation fold fits preprocessing only on that fold's training rows to avoid information leakage.

In [ ]:
def add_features(frame):
    """Create derived customer tenure, usage, support, payment, and loyalty features."""
    data = frame.copy()
    data['tenure_bucket'] = pd.cut(
        data['tenure_months'], [0, 6, 24, 48, np.inf],
        labels=['New', 'Growing', 'Mature', 'Loyal'],
    ).astype('object')
    data['charges_per_month'] = data['total_charges'] / data['tenure_months'].clip(lower=1)
    login_norm = data['login_count_30d'].clip(0, 60) / 60
    data['usage_trend'] = (login_norm + data['session_minutes_30d'].fillna(0).clip(0, 2000) / 2000) / 2
    data['engagement_score'] = (data['feature_usage_score'].fillna(50) / 100 + login_norm) / 2 * 100
    data['support_load'] = data['num_support_tickets_30d'].fillna(0).clip(0, 15) * np.log1p(
        data['avg_resolution_time_hours'].fillna(data['avg_resolution_time_hours'].median()).clip(0, 96)
    )
    data['payment_reliability'] = 1 - data['num_payment_failures_6m'].fillna(0).clip(0, 5) / 5
    data['recency_score'] = 1 / (1 + data['days_since_last_login'].fillna(30).clip(0, 180) / 30)
    data['loyalty_score'] = (
        data['tenure_months'].clip(0, 72) / 72 * .5
        + data['referral_count'].clip(0, 10) / 10 * .3
        + data['auto_pay'].fillna(0) * .2
    ) * 100
    return data

NUMERIC_COLS = [
    'tenure_months', 'monthly_charges', 'total_charges', 'num_support_tickets_30d',
    'avg_resolution_time_hours', 'satisfaction_score', 'login_count_30d',
    'session_minutes_30d', 'feature_usage_score', 'days_since_last_login',
    'num_payment_failures_6m', 'referral_count', 'charges_per_month',
    'usage_trend', 'engagement_score', 'support_load', 'payment_reliability',
    'recency_score', 'loyalty_score',
]
CAT_COLS = ['plan_type', 'contract_type', 'payment_method', 'tenure_bucket']
BINARY_COLS = ['auto_pay', 'discount_applied', 'price_increase_last_3m']
TARGET = 'churn'

X = customers.drop(columns=['customer_id', 'signup_date', TARGET])
y = customers[TARGET].astype(int)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=.20, random_state=42, stratify=y,
)

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
binary_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('scaler', StandardScaler()),
])

def build_preprocessor():
    return ColumnTransformer([
        ('numeric', numeric_pipeline, NUMERIC_COLS),
        ('categorical', categorical_pipeline, CAT_COLS),
        ('binary', binary_pipeline, BINARY_COLS),
    ], remainder='drop')

print(f'Train: {len(X_train_raw):,} | Holdout: {len(X_test_raw):,}')
print(f'Train churn rate: {y_train.mean():.1%} | Holdout churn rate: {y_test.mean():.1%}')
display(add_features(X_train_raw.head())[list(NUMERIC_COLS[-7:])].head())

## 3. Candidate models and validation

The training module compares Logistic Regression, Random Forest, XGBoost, and LightGBM with five-fold `StratifiedKFold`. It reports ROC-AUC and recall at a 0.40 threshold, then selects the best candidate by mean ROC-AUC. Class imbalance is addressed with estimator-specific class weights.

To retrain all candidates, set `RUN_FULL_TRAINING = True`. This is computationally heavier than the remaining notebook cells. If a saved model is absent, training runs automatically.

In [ ]:
from sklearn.base import clone

RUN_FULL_TRAINING = True
SEED = 42
N_FOLDS = 5

negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
positive_weight = negative_count / max(positive_count, 1)
model_factories = {
    'LogisticRegression': lambda: LogisticRegression(
        class_weight='balanced', max_iter=1000, C=.5, random_state=SEED,
    ),
    'RandomForest': lambda: RandomForestClassifier(
        n_estimators=160, class_weight='balanced', max_depth=12,
        min_samples_leaf=5, n_jobs=-1, random_state=SEED,
    ),
    'XGBoost': lambda: XGBClassifier(
        n_estimators=180, learning_rate=.05, max_depth=5,
        scale_pos_weight=positive_weight, subsample=.8, colsample_bytree=.8,
        eval_metric='logloss', random_state=SEED, n_jobs=-1, verbosity=0,
    ),
    'LightGBM': lambda: LGBMClassifier(
        n_estimators=180, learning_rate=.05, max_depth=6,
        is_unbalance=True, subsample=.8, colsample_bytree=.8,
        random_state=SEED, verbosity=-1,
    ),
}

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
cv_results = {}
for model_name, factory in model_factories.items():
    fold_auc, fold_recall = [], []
    print(f'Validating {model_name}...')
    for fold_number, (train_idx, valid_idx) in enumerate(cv.split(X_train_raw, y_train), 1):
        fold_preprocessor = build_preprocessor()
        fold_train = add_features(X_train_raw.iloc[train_idx])
        fold_valid = add_features(X_train_raw.iloc[valid_idx])
        transformed_train = fold_preprocessor.fit_transform(fold_train)
        transformed_valid = fold_preprocessor.transform(fold_valid)
        fold_model = factory()
        fold_model.fit(transformed_train, y_train.iloc[train_idx])
        probability = fold_model.predict_proba(transformed_valid)[:, 1]
        fold_auc.append(roc_auc_score(y_train.iloc[valid_idx], probability))
        fold_recall.append(recall_score(y_train.iloc[valid_idx], probability >= .40, zero_division=0))
        print(f'  Fold {fold_number}: ROC-AUC={fold_auc[-1]:.3f}, recall={fold_recall[-1]:.3f}')
    cv_results[model_name] = {
        'roc_auc': float(np.mean(fold_auc)),
        'recall_at_0_40': float(np.mean(fold_recall)),
    }

cv_table = pd.DataFrame(cv_results).T.sort_values('roc_auc', ascending=False)
display(cv_table.style.format('{:.3f}'))
best_name = cv_table.index[0]

preprocessor = build_preprocessor()
X_train_transformed = preprocessor.fit_transform(add_features(X_train_raw))
X_test_transformed = preprocessor.transform(add_features(X_test_raw))
model = model_factories[best_name]()
model.fit(X_train_transformed, y_train)
threshold = .40
bundle = {
    'model': model,
    'preprocessor': preprocessor,
    'model_name': best_name,
    'feature_names': list(preprocessor.get_feature_names_out()),
    'threshold': threshold,
    'cv_results': cv_results,
}
joblib.dump(bundle, MODEL_PATH)
print(f'Selected {best_name} by mean CV ROC-AUC; saved model bundle to {MODEL_PATH}')

## 3. Candidate models and validation

The notebook compares Logistic Regression, Random Forest, XGBoost, and LightGBM with five-fold `StratifiedKFold`. It reports mean ROC-AUC and recall at a 0.40 threshold, then selects the candidate with the highest mean ROC-AUC. Class imbalance is handled with each estimator's class-weighting options; duplicate SMOTE resampling is avoided.

Running the training cell fits all four models and can take several minutes. The selected model and preprocessing pipeline are saved together as `models/churn_model.pkl`.

In [ ]:
test_probability = model.predict_proba(X_test_transformed)[:, 1]
test_prediction = (test_probability >= threshold).astype(int)
confusion = confusion_matrix(y_test, test_prediction)
metrics = {
    'accuracy': accuracy_score(y_test, test_prediction),
    'precision': precision_score(y_test, test_prediction, zero_division=0),
    'recall': recall_score(y_test, test_prediction, zero_division=0),
    'f1': f1_score(y_test, test_prediction, zero_division=0),
    'roc_auc': roc_auc_score(y_test, test_probability),
    'pr_auc': average_precision_score(y_test, test_probability),
}
print(f'Model: {best_name} | Classification threshold: {threshold:.2f}')
display(pd.Series(metrics, name='holdout').to_frame().style.format('{:.4f}'))
print('Confusion matrix (rows=actual, columns=predicted):')
display(pd.DataFrame(confusion, index=['Actual retained', 'Actual churn'],
                     columns=['Predicted retained', 'Predicted churn']))
print(classification_report(y_test, test_prediction,
                            target_names=['Retained', 'Churned'], zero_division=0))

In [ ]:
false_positive_rate, true_positive_rate, _ = roc_curve(y_test, test_probability)
precision_values, recall_values, _ = precision_recall_curve(y_test, test_probability)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(false_positive_rate, true_positive_rate, label=f"ROC-AUC = {metrics['roc_auc']:.3f}")
axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[0].set(xlabel='False positive rate', ylabel='True positive rate', title='ROC curve')
axes[0].legend()
axes[1].plot(recall_values, precision_values, label=f"PR-AUC = {metrics['pr_auc']:.3f}", color='#C0504D')
axes[1].axhline(y_test.mean(), linestyle='--', color='gray', label='Prevalence baseline')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-recall curve')
axes[1].legend()
plt.tight_layout()
plt.show()

ranked = pd.DataFrame({'actual': y_test.to_numpy(), 'probability': test_probability})
ranked = ranked.sort_values('probability', ascending=False).reset_index(drop=True)
ranked['decile'] = np.ceil((np.arange(len(ranked)) + 1) / len(ranked) * 10).astype(int).clip(1, 10)
overall_churn = ranked['actual'].mean()
deciles = ranked.groupby('decile').agg(customers=('actual', 'size'), churners=('actual', 'sum'))
deciles['churn_rate'] = deciles['churners'] / deciles['customers']
deciles['lift'] = deciles['churn_rate'] / overall_churn
deciles['cumulative_churn_rate'] = deciles['churners'].cumsum() / deciles['customers'].cumsum()
display(deciles.round(3))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(deciles.index, deciles['lift'], color='#2878A0')
ax.axhline(1, color='#C0504D', linestyle='--', label='Baseline lift')
ax.set(xlabel='Risk decile (1 = highest predicted risk)', ylabel='Lift', title='Churn lift by risk decile')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Explainability and churn drivers

SHAP is used to explain model outputs globally and for individual customers. Global importance summarizes which features most influence predictions in this model; an individual explanation ranks that customer's largest contributing features. These are model associations, not proof that changing a feature will prevent churn.

In [ ]:
feature_names = bundle['feature_names']
shap_sample = np.asarray(X_test_transformed[:min(500, len(X_test_transformed))])
if best_name in {'RandomForest', 'XGBoost', 'LightGBM'}:
    shap_explainer = shap.TreeExplainer(model)
else:
    shap_explainer = shap.LinearExplainer(model, X_train_transformed[:min(500, len(X_train_transformed))])
shap_output = shap_explainer.shap_values(shap_sample)
if isinstance(shap_output, list):
    global_shap_values = np.asarray(shap_output[1])
else:
    global_shap_values = np.asarray(shap_output)
    if global_shap_values.ndim == 3:
        global_shap_values = global_shap_values[:, :, 1]

mean_abs_shap = np.abs(global_shap_values).mean(axis=0)
importance = pd.DataFrame({'feature': feature_names, 'mean_abs_shap': mean_abs_shap})
importance = importance.sort_values('mean_abs_shap', ascending=False).head(20)
display(importance)
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(importance['feature'][::-1], importance['mean_abs_shap'][::-1], color='#2878A0')
ax.set(xlabel='Mean absolute SHAP value', title='Global churn feature importance')
plt.tight_layout()
plt.show()

# Beeswarm plot provides direction and distribution of feature contributions.
shap.summary_plot(global_shap_values, shap_sample, feature_names=feature_names,
                  max_display=15, show=False)
plt.tight_layout()
plt.show()

# SHAP contribution reason codes for high-risk customers in the explained sample.
sample_probability = test_probability[:len(global_shap_values)]
high_risk_positions = np.flatnonzero(sample_probability >= .65)
reason_rows = []
for position in high_risk_positions:
    values = global_shap_values[position]
    top_positions = np.argsort(values)[::-1][:3]
    reason_rows.append({
        'holdout_position': int(position),
        'churn_probability': float(sample_probability[position]),
        **{f'reason_{rank + 1}': feature_names[feature_idx]
           for rank, feature_idx in enumerate(top_positions)},
    })
reason_codes = pd.DataFrame(reason_rows)
display(reason_codes.head(10))

## 6. Score a customer and recommend an action

The example below scores one customer using the same feature and preprocessing pipeline used during training. SHAP contributions become reason codes, then the retention engine maps the risk profile to an offer and estimates its cost, expected saved revenue, and ROI. The estimates use assumed offer success rates and should be validated with controlled campaigns.

In [ ]:
OFFERS = {
    'DISCOUNT': {'description': '10-15% discount for three months', 'retention': .68, 'cost_rate': .125, 'months': 3},
    'FREE_MONTH': {'description': 'One free month credited to the account', 'retention': .62, 'cost_rate': 1.0, 'months': 1},
    'PLAN_UPGRADE': {'description': 'Complimentary plan upgrade for three months', 'retention': .72, 'cost_rate': .20, 'months': 3},
    'PRIORITY_SUPPORT': {'description': 'Priority support with a response target under two hours', 'retention': .55, 'cost_rate': .05, 'months': 3},
    'PERSONAL_CALL': {'description': 'Personal account review call', 'retention': .50, 'cost_rate': .02, 'months': 1},
    'LOYALTY_REWARD': {'description': 'Loyalty points and referral reward', 'retention': .45, 'cost_rate': .08, 'months': 1},
}

def choose_offer(probability, reasons, customer_data):
    reason_text = ' '.join(reasons).lower()
    if any(term in reason_text for term in ['price_increase', 'monthly_charges', 'charges_per_month']):
        return 'DISCOUNT'
    if customer_data.get('plan_type') == 'Basic' and customer_data.get('monthly_charges', 0) >= 35:
        return 'PLAN_UPGRADE'
    if any(term in reason_text for term in ['support', 'satisfaction', 'resolution']):
        return 'PRIORITY_SUPPORT'
    if any(term in reason_text for term in ['usage', 'engagement', 'login', 'recency', 'session']):
        return 'FREE_MONTH'
    if any(term in reason_text for term in ['payment', 'reliability', 'failure']):
        return 'DISCOUNT'
    if customer_data.get('tenure_months', 0) >= 24:
        return 'LOYALTY_REWARD'
    return 'PERSONAL_CALL' if probability >= .65 else 'LOYALTY_REWARD'

def estimate_offer(probability, reasons, customer_data):
    offer_type = choose_offer(probability, reasons, customer_data)
    offer = OFFERS[offer_type]
    monthly_revenue = float(customer_data.get('monthly_charges', 50))
    cost = monthly_revenue * offer['cost_rate'] * offer['months']
    expected_saved = probability * offer['retention'] * monthly_revenue * 12
    roi = (expected_saved - cost) / cost * 100 if cost else 0.0
    return {
        'offer_type': offer_type,
        'description': offer['description'],
        'estimated_retention_probability': offer['retention'],
        'offer_cost': round(cost, 2),
        'expected_revenue_saved': round(expected_saved, 2),
        'roi_pct': round(roi, 1),
    }

example_index = y_test.index[0]
example = customers.loc[example_index]
example_features = example.drop(labels=['customer_id', 'signup_date', 'churn']).to_frame().T
example_transformed = preprocessor.transform(add_features(example_features))
example_probability = float(model.predict_proba(example_transformed)[0, 1])
example_shap = shap_explainer.shap_values(np.asarray(example_transformed))
if isinstance(example_shap, list):
    example_shap = np.asarray(example_shap[1])[0]
else:
    example_shap = np.asarray(example_shap)
    if example_shap.ndim == 3:
        example_shap = example_shap[0, :, 1]
    else:
        example_shap = example_shap[0]
example_top_idx = np.argsort(example_shap)[::-1][:5]
example_reasons = [feature_names[idx] for idx in example_top_idx]
example_offer = estimate_offer(example_probability, example_reasons, example_features.iloc[0].to_dict())
example_risk = 'high' if example_probability >= .65 else 'medium' if example_probability >= .40 else 'low'

print(f"Customer: {example['customer_id']}")
print(f'Churn probability: {example_probability:.1%} ({example_risk} risk)')
display(pd.DataFrame({
    'reason': [feature_names[idx] for idx in example_top_idx],
    'shap_contribution': [round(float(example_shap[idx]), 4) for idx in example_top_idx],
}))
print(f"Recommended offer: {example_offer['offer_type']} - {example_offer['description']}")
print(f"Estimated offer cost: ${example_offer['offer_cost']:,.2f}")
print(f"Expected revenue saved (12-month proxy): ${example_offer['expected_revenue_saved']:,.2f}")
print(f"Estimated ROI: {example_offer['roi_pct']:,.1f}%")

## 7. Campaign economics

For customer $i$, the project's expected-revenue estimate is `churn_probability_i x assumed_offer_retention_probability_i x monthly_revenue_i x 12`. Estimated campaign ROI is `(total_expected_revenue_saved - total_offer_cost) / total_offer_cost x 100`. This is an illustrative prioritization estimate, not incremental lift: it assumes the offer's retention probability and does not estimate what would have happened without the campaign. Use a randomized holdout to measure causal campaign impact before committing budget.

In [ ]:
all_features = customers.drop(columns=['customer_id', 'signup_date', 'churn'])
all_transformed = preprocessor.transform(add_features(all_features))
all_probability = model.predict_proba(all_transformed)[:, 1]
priority_indices = np.flatnonzero(all_probability >= .65)
priority_indices = priority_indices[np.argsort(all_probability[priority_indices])[::-1]][:100]

campaign = []
for idx in priority_indices:
    row = customers.iloc[idx]
    customer_data = row.drop(labels=['customer_id', 'signup_date', 'churn']).to_dict()
    campaign.append(estimate_offer(float(all_probability[idx]), ['high_risk_profile'], customer_data))

if campaign:
    campaign_cost = sum(item['offer_cost'] for item in campaign)
    expected_saved = sum(item['expected_revenue_saved'] for item in campaign)
    mrr_at_risk = sum(
        float(customers.iloc[idx]['monthly_charges']) * float(all_probability[idx])
        for idx in priority_indices
    )
    campaign_roi = (expected_saved - campaign_cost) / campaign_cost * 100 if campaign_cost else 0.0
    campaign_summary = pd.Series({
        'customers_targeted': len(campaign),
        'monthly_revenue_at_risk': round(mrr_at_risk, 2),
        'estimated_offer_cost': round(campaign_cost, 2),
        'expected_revenue_saved_12m': round(expected_saved, 2),
        'expected_net_benefit': round(expected_saved - campaign_cost, 2),
        'estimated_campaign_roi_pct': round(campaign_roi, 1),
    }, name='Top-risk campaign')
    display(campaign_summary.to_frame())
else:
    print('No customers exceed the high-risk threshold in this dataset.')

print('ROI is an assumption-based estimate, not a measured causal campaign effect.')

## 8. Limitations, deployment, and conclusion

- The dataset and outcomes are synthetic. These metrics demonstrate the workflow, not production accuracy. Validate on representative historical customer data before deployment.
- The 0.40 classification threshold emphasizes recall, which can create substantial false-positive outreach. Campaign capacity, offer cost, and customer experience should inform the operating threshold.
- SHAP values describe model behavior, not causal drivers. Offer-retention rates and revenue saved are assumptions until measured experimentally.
- Monitor calibration, performance by customer segment, data quality, and drift. Avoid using protected or inappropriate attributes.

**Deployment handoff:** this single-file submission demonstrates the analytical and scoring workflow. In a production system, package the same fitted preprocessing/model bundle behind a FastAPI endpoint and provide customer-success users a Streamlit dashboard for filtering alerts, scoring new customers, and reviewing model insights. API and dashboard services require a running Python environment and are separate runtime processes from this notebook.

**Conclusion:** the notebook demonstrates an end-to-end decision-support workflow: generate and inspect data, engineer features, compare models, rank churn risk, explain predictions, recommend a targeted offer, and estimate campaign economics. Production use requires real customer data, validation, and controlled measurement of incremental retention.